In [ ]:
import shutil
import socket
from dataclasses import dataclass
from datetime import datetime, timezone
from math import prod
from pathlib import Path
from pprint import pprint
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed
from memory_tempfile import MemoryTempfile as MemoryTemper
from tifffile import TiffFile

from comptest import ProcessResult, imagemagick, libtiff, run_process
from comptest.constants import DATASETS_DIR
from comptest.fs import get_dir_size_bytes
from comptest.types import CompressionTest

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
PROJECT_DIR = Path().resolve()
OUTPUT_DIR = PROJECT_DIR / "output"
assert OUTPUT_DIR.is_dir()

mem_temper = MemoryTemper(filesystem_types=["tmpfs"])

In [ ]:
COMPRESSION_TESTS_DIR = Path(
    "/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests"
)
assert COMPRESSION_TESTS_DIR.is_dir()

In [ ]:
input_file = COMPRESSION_TESTS_DIR / "uncompressed.tif"
assert input_file.is_file()

output_file = COMPRESSION_TESTS_DIR / "compressed.tif"

In [ ]:
run_process("tiffcp", ["-c", "lzma:2:p6", str(input_file), str(output_file)])

ProcessResult(time=ProcessTiming(wall=2.4991967359092087, user=2.429042, system=0.036955), memory=ProcessMemoryUsage(max_rss=211951616), returncode=0)

In [ ]:
DATASET = "Canislupus_CZM_712"
DATASET_DIR = DATASETS_DIR / DATASET
assert DATASET_DIR.is_dir()

SCRATCH_INPUT_FILES = list(DATASET_DIR.glob("*.tif"))

In [ ]:
# Load input files into memory
# !!! Note: We currently don't clean this up after use!
temp_dir_path = (
    Path(mem_temper.gettempdir()) / "s250250-compression-tests-filestack-in-memory"
)

if temp_dir_path.exists():
    shutil.rmtree(temp_dir_path)

temp_dir_path.mkdir(parents=True, exist_ok=True)
INPUT_FILES = []
for file in SCRATCH_INPUT_FILES:
    shutil.copy(file, temp_dir_path / file.name)
    INPUT_FILES.append(temp_dir_path / file.name)


In [ ]:
# Check dtype is the same for all
INPUT_DTYPE = None
for file in INPUT_FILES:
    with TiffFile(file) as tif:
        dtype = tif.pages[0].dtype
        if INPUT_DTYPE is None:
            INPUT_DTYPE = dtype
        else:
            assert dtype == INPUT_DTYPE, (
                f"Input files have different dtypes: {dtype} != {INPUT_DTYPE}"
            )
INPUT_DTYPE

dtype('uint16')

In [ ]:
DTYPE_UINT = {
    np.bool,
    np.uint8,
    np.uint16,
    np.uint32,
    np.uint64,
}
DTYPE_SINT = {
    np.int8,
    np.int16,
    np.int32,
    np.int64,
}
DTYPE_INT = DTYPE_UINT.union(DTYPE_SINT)
DTYPE_FLOAT = {
    np.float16,
    np.float32,
    np.float64,
    np.float128,
}
DTYPE_COMPLEX = {
    np.complex64,
    np.complex128,
    np.complex256,
}
DTYPE_ANY = DTYPE_INT.union(DTYPE_FLOAT).union(DTYPE_COMPLEX)

In [ ]:
# TESTS DEFINITIONS

# Uncomment below to see options for tiffcp
# !tiffcp --help
TESTS = {
    # ============================== N-BIT LOSSLESS ==============================
    # --- ZIP / DEFLATE ---
    # --- LZMA ---
    "lzma_pred1_p0": {
        "libtiff": ["-c", "lzma:1:p0"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "0",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p2": {
        "libtiff": ["-c", "lzma:1:p2"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "20",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p4": {
        "libtiff": ["-c", "lzma:1:p4"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "40",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p6": {
        "libtiff": ["-c", "lzma:1:p6"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p8": {
        "libtiff": ["-c", "lzma:1:p8"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "80",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p9": {
        "libtiff": ["-c", "lzma:1:p9"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p0": {
        "libtiff": ["-c", "lzma:2:p0"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "0",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p2": {
        "libtiff": ["-c", "lzma:2:p2"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "20",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p4": {
        "libtiff": ["-c", "lzma:2:p4"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "40",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p6": {
        "libtiff": ["-c", "lzma:2:p6"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p8": {
        "libtiff": ["-c", "lzma:2:p8"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "80",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p9": {
        "libtiff": ["-c", "lzma:2:p9"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    # --- ZSTD ---
    "zstd_pred1_p1": {
        "libtiff": ["-c", "zstd:1:p1"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "5",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p5": {
        "libtiff": ["-c", "zstd:1:p5"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "23",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p9": {
        "libtiff": ["-c", "zstd:1:p9"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "41",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p13": {
        "libtiff": ["-c", "zstd:1:p13"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "59",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p17": {
        "libtiff": ["-c", "zstd:1:p17"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "77",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p19": {
        "libtiff": ["-c", "zstd:1:p19"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "86",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p22": {
        "libtiff": ["-c", "zstd:1:p22"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "100",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p1": {
        "libtiff": ["-c", "zstd:2:p1"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "5",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p5": {
        "libtiff": ["-c", "zstd:2:p5"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "23",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p9": {
        "libtiff": ["-c", "zstd:2:p9"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "41",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p13": {
        "libtiff": ["-c", "zstd:2:p13"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "59",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p17": {
        "libtiff": ["-c", "zstd:2:p17"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "77",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p19": {
        "libtiff": ["-c", "zstd:2:p19"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "86",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p22": {
        "libtiff": ["-c", "zstd:2:p22"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "100",
        ],
        "notes": ["lossless"],
    },
    # ============================== N-BIT LOSSY (LERC z>0) ==============================
    "lerc_z0.5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:0.5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z1_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:1:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z2_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:2:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z10_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:10:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z20_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:20:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    # ============================== 8-BIT LOSSLESS ==============================
    # libtiff: None -- tiffcp requires 8-bit source data already; can't downsample inline
    "webp_lossless_pred1_p50": {
        "libtiff": None,  # WEBP codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "50",
        ],
        "notes": ["lossless", "8bit"],
    },
    "webp_lossless_pred1_p75": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "75",
        ],
        "notes": ["lossless", "8bit"],
    },
    "webp_lossless_pred1_p100": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "100",
        ],
        "notes": ["lossless", "8bit"],
    },
    "g3_1d": {
        "libtiff": None,  # G3 is bilevel-only; source is 16-bit grayscale
        "imagemagick": ["-depth", "8", "-monochrome", "-compress", "Fax"],
        "notes": ["lossless", "1bit"],
    },
    "g3_2d": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-monochrome",
            "-compress",
            "Fax",
        ],  # 1D/2D not separately exposed
        "notes": ["lossless", "1bit"],
    },
    "g3_2d_fill": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-monochrome",
            "-compress",
            "Fax",
        ],  # fill/EOL byte-align not exposed
        "notes": ["lossless", "1bit"],
    },
    "g4": {
        "libtiff": None,  # G4 is bilevel-only; source is 16-bit grayscale
        "imagemagick": ["-depth", "8", "-monochrome", "-compress", "Group4"],
        "notes": ["lossless", "1bit"],
    },
    # ============================== 8-BIT LOSSY ==============================
    "webp_pred2_p25": {
        "libtiff": None,  # WEBP codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "25"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p50": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "50"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p65": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "65"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p75": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "75"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p90": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "90"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p100": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "100"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q10": {
        "libtiff": None,  # JPEG codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "10"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q25": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "25"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q50": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "50"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q65": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "65"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q75": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "75"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q85": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "85"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q95": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "95"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q96": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "96"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q97": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "97"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q98": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "98"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q99": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "99"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q100": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "100"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q50_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "50",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q75_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "75",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q85_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "85",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q95_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "95",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
}

In [ ]:
# TESTS DEFINITIONS

# Uncomment below to see options for tiffcp
# !tiffcp --help
TESTS = {
    # ============================== N-BIT LOSSLESS ==============================
    # --- ZIP / DEFLATE ---
    "zip_pred1_p1": {
        "libtiff": ["-c", "zip:1:p1"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "10",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p3": {
        "libtiff": ["-c", "zip:1:p3"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "30",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p5": {
        "libtiff": ["-c", "zip:1:p5"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "50",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p6": {
        "libtiff": ["-c", "zip:1:p6"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p7": {
        "libtiff": ["-c", "zip:1:p7"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "70",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p9": {
        "libtiff": ["-c", "zip:1:p9"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p1": {
        "libtiff": ["-c", "zip:2:p1"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "10",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p3": {
        "libtiff": ["-c", "zip:2:p3"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "30",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p5": {
        "libtiff": ["-c", "zip:2:p5"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "50",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p6": {
        "libtiff": ["-c", "zip:2:p6"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p7": {
        "libtiff": ["-c", "zip:2:p7"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "70",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p9": {
        "libtiff": ["-c", "zip:2:p9"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p9_s0_zlib": {
        "libtiff": ["-c", "zip:2:p9:s0"],
        "imagemagick": None,  # ImageMagick has no zlib/libdeflate subcodec switch
        "notes": ["lossless"],
    },
    "zip_pred2_p9_s1_libdeflate": {
        "libtiff": ["-c", "zip:2:p9:s1"],
        "imagemagick": None,  # ImageMagick has no zlib/libdeflate subcodec switch
        "notes": ["lossless"],
    },
    # --- LZMA ---
    "lzma_pred1_p0": {
        "libtiff": ["-c", "lzma:1:p0"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "0",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p2": {
        "libtiff": ["-c", "lzma:1:p2"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "20",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p4": {
        "libtiff": ["-c", "lzma:1:p4"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "40",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p6": {
        "libtiff": ["-c", "lzma:1:p6"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p8": {
        "libtiff": ["-c", "lzma:1:p8"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "80",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p9": {
        "libtiff": ["-c", "lzma:1:p9"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p0": {
        "libtiff": ["-c", "lzma:2:p0"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "0",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p2": {
        "libtiff": ["-c", "lzma:2:p2"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "20",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p4": {
        "libtiff": ["-c", "lzma:2:p4"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "40",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p6": {
        "libtiff": ["-c", "lzma:2:p6"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p8": {
        "libtiff": ["-c", "lzma:2:p8"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "80",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p9": {
        "libtiff": ["-c", "lzma:2:p9"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    # --- ZSTD ---
    "zstd_pred1_p1": {
        "libtiff": ["-c", "zstd:1:p1"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "5",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p5": {
        "libtiff": ["-c", "zstd:1:p5"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "23",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p9": {
        "libtiff": ["-c", "zstd:1:p9"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "41",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p13": {
        "libtiff": ["-c", "zstd:1:p13"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "59",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p17": {
        "libtiff": ["-c", "zstd:1:p17"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "77",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p19": {
        "libtiff": ["-c", "zstd:1:p19"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "86",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p22": {
        "libtiff": ["-c", "zstd:1:p22"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "100",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p1": {
        "libtiff": ["-c", "zstd:2:p1"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "5",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p5": {
        "libtiff": ["-c", "zstd:2:p5"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "23",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p9": {
        "libtiff": ["-c", "zstd:2:p9"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "41",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p13": {
        "libtiff": ["-c", "zstd:2:p13"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "59",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p17": {
        "libtiff": ["-c", "zstd:2:p17"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "77",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p19": {
        "libtiff": ["-c", "zstd:2:p19"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "86",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p22": {
        "libtiff": ["-c", "zstd:2:p22"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "100",
        ],
        "notes": ["lossless"],
    },
    # --- LERC (z0 = lossless) ---
    "lerc_z0_s0_none": {
        "libtiff": ["-c", "lerc:0:s0"],
        "imagemagick": None,  # no LERC coder in ImageMagick
        "notes": ["lossless"],
    },
    "lerc_z0_s1_deflate_p1": {
        "libtiff": ["-c", "lerc:0:s1:p1"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s1_deflate_p5": {
        "libtiff": ["-c", "lerc:0:s1:p5"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s1_deflate_p9": {
        "libtiff": ["-c", "lerc:0:s1:p9"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p1": {
        "libtiff": ["-c", "lerc:0:s2:p1"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p9": {
        "libtiff": ["-c", "lerc:0:s2:p9"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p13": {
        "libtiff": ["-c", "lerc:0:s2:p13"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p17": {
        "libtiff": ["-c", "lerc:0:s2:p17"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:0:s2:p22"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    # --- PackBits ---
    "packbits": {
        "libtiff": ["-c", "packbits"],
        "imagemagick": ["-compress", "RLE"],
        "notes": ["lossless"],
    },
    # --- SGILOG ---
    "sgilog": {
        "libtiff": ["-c", "sgilog"],
        "imagemagick": None,
        "notes": ["lossless", "designed-for-logluv-hdr"],
    },
    # ============================== N-BIT LOSSY (LERC z>0) ==============================
    "lerc_z0.5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:0.5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z1_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:1:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z2_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:2:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z10_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:10:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z20_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:20:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
}

In [ ]:
from comptest.types import FileStackCompressionConfig

_C = FileStackCompressionConfig

BASELINE_CONFIGS = [
    _C(
        name="libtiff_baseline",
        series="tiff-baseline",
        vary_var=None,
        vary_value=None,
        is_lossless=True,
        groupings={
            "implementation": "libtiff/tiffcp",
        },
        allowed_dtypes=DTYPE_ANY,
        compressor=(libtiff.compress, ("-c", "none"), {}),
    ),
    _C(
        name="imagemagick_baseline",
        series="tiff-baseline",
        vary_var=None,
        vary_value=None,
        is_lossless=True,
        groupings={
            "implementation": "imagemagick",
        },
        allowed_dtypes=DTYPE_ANY,
        compressor=(imagemagick, ("-compress", "None"), {}),
    ),
]

_PREDICTOR_NAME_MAP = {1: "none", 2: "horizontal", 3: "float"}
LIBTIFF_LZW_CONFIGS = [
    _C(
        name="libtiff_lzw_pred{i}",
        series="tiff-lzw",
        vary_var="predictor",
        vary_value=i,
        is_lossless=True,
        groupings={
            "implementation": "libtiff/tiffcp",
            "differencing": _PREDICTOR_NAME_MAP[i],
            "predictor": i,
        },
        allowed_dtypes=DTYPE_ANY if i == 3 else DTYPE_INT,
        compressor=(libtiff.compress, ("-c", f"lzw:{i}"), {}),
    )
    for i in [1, 2, 3]  # 1: no differencing
]
IMAGEMAGICK_LZW_CONFIGS = [
    _C(
        name="imagemagick_lzw_pred{i}",
        series="tiff-lzw",
        vary_var="predictor",
        vary_value=i,
        is_lossless=True,
        groupings={
            "implementation": "imagemagick",
            "differencing": _PREDICTOR_NAME_MAP[i],
            "predictor": i,
        },
        allowed_dtypes=DTYPE_ANY if i == 3 else DTYPE_INT,
        compressor=(
            imagemagick,
            ("-compress", "LZW", "-define", f"tiff:predictor={i}"),
            {},
        ),
    )
    for i in [1, 2, 3]
]

LZW_CONFIGS = LIBTIFF_LZW_CONFIGS + IMAGEMAGICK_LZW_CONFIGS

LIBTIFF_DEFLATE_CONFIGS = [
    _C(
        name=f"libtiff_deflate_pred{i}_s{s}_p{p}",
        series="tiff-deflate",
        vary_var="effort",
        vary_value=p,
        is_lossless=True,
        groupings={
            "implementation": "libtiff/tiffcp",
            "differencing": _PREDICTOR_NAME_MAP[i],
            "predictor": i,
            "effort": p,
            "subcodec": "zlib" if s == 0 else "libdeflate",
            "subcodec_index": s,
        },
        allowed_dtypes=DTYPE_ANY if i == 3 else DTYPE_INT,
        compressor=(libtiff.compress, ("-c", f"zip:{i}:s{s}:p{p}"), {}),
    )
    for i in [1, 2, 3]
    for p in [1, 2, 3, 4, 5, 6, 7, 8, 9]
    for s in [0, 1]  # subcodec, 0=zlib, 1=libdeflate
]

IMAGEMAGICK_DEFLATE_CONFIGS = [
    _C(
        name=f"imagemagick_deflate_pred{i}_p{p}",
        series="tiff-deflate",
        vary_var="effort",
        vary_value=p,
        is_lossless=True,
        groupings={
            "implementation": "imagemagick",
            "differencing": _PREDICTOR_NAME_MAP[i],
            "predictor": i,
            "effort": p,
            "subcodec": None,
        },
        allowed_dtypes=DTYPE_ANY if i == 3 else DTYPE_INT,
        compressor=(
            imagemagick,
            (
                "-compress",
                "Zip",
                "-define",
                f"tiff:predictor={i}",
                "-quality",
                f"{p * 10}",
            ),
            {},
        ),
    )
    for i in [1, 2, 3]
    for p in [1, 2, 3, 4, 5, 6, 7, 8, 9]
    # Note: No subcodec available in ImageMagick, so no s=0/1 loop here
]

DEFLATE_CONFIGS = LIBTIFF_DEFLATE_CONFIGS + IMAGEMAGICK_DEFLATE_CONFIGS


_LIBTIFF_LERC_SUBCODECS = {
    0: "none",
    1: "deflate",
    2: "zstd",
}

_LIBTIFF_LERC_COMBINATIONS = []

for p in [1, 5, 8, 10, 12, 14, 16, 18, 20, 22]:
    for s in [0, 1, 2]:
        if s == 1 and p > 12:
            continue  # deflate subcodec only supports p <= 12 for whatever reason, I think a bug in libtiff
        _LIBTIFF_LERC_COMBINATIONS.append({"z": 0, "s": s, "p": p})

LIBTIFF_LERC_CONFIGS = [
    _C(
        name=f"libtiff_lerc_effort_z{c['z']}_s{c['s']}_p{c['p']}",
        series="tiff-lerc",
        vary_var="effort",
        vary_value=c["p"],
        is_lossless=True,
        groupings={
            "implementation": "libtiff/tiffcp",
            "subcodec": _LIBTIFF_LERC_SUBCODECS[c["s"]],
            "effort": c["p"],
            "subcodec_index": c["s"],
            "z_max_error": c["z"],
        },
        allowed_dtypes=DTYPE_ANY,
        compressor=(libtiff.compress, ("-c", f"lerc:{c['z']}:s{c['s']}:p{c['p']}"), {}),
    )
    for c in _LIBTIFF_LERC_COMBINATIONS
]

LIBTIFF_SGILOG_CONFIGS = [
    _C(
        name="libtiff_sgilog",
        series="tiff-sgilog",
        vary_var=None,
        vary_value=None,
        is_lossless=True,
        groupings={
            "implementation": "libtiff/tiffcp",
        },
        allowed_dtypes=DTYPE_ANY,
        compressor=(libtiff.compress, ("-c", "sgilog"), {}),
    )
]

RLE_CONFIGS = [
    _C(
        name="libtiff_rle",
        series="tiff-rle",
        vary_var=None,
        vary_value=None,
        is_lossless=True,
        groupings={
            "implementation": "libtiff/tiffcp",
        },
        allowed_dtypes=DTYPE_ANY,
        compressor=(libtiff.compress, ("-c", "packbits"), {}),
    ),
    _C(
        name="imagemagick_rle",
        series="tiff-rle",
        vary_var=None,
        vary_value=None,
        is_lossless=True,
        groupings={
            "implementation": "imagemagick",
        },
        allowed_dtypes=DTYPE_ANY,
        compressor=(imagemagick, ("-compress", "RLE"), {}),
    ),
]

# TODO: u1/u8 codeccs not set up since we don't really care about those

CONFIGS = [] + LIBTIFF_LERC_CONFIGS + LIBTIFF_SGILOG_CONFIGS

NameError: name '_PREDICTOR_NAM' is not defined

In [ ]:
BASE_EXTRA = {
    "n_jobs": 16,
}

host = socket.gethostname()
initial_size = sum(f.stat().st_size for f in INPUT_FILES)

tests = []

TEST_ID = "full"
TEST_OUTPUT_DIR = (
    OUTPUT_DIR
    / "file-stack"
    / (TEST_ID + "_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    / DATASET
)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# for config in CONFIGS:
for config in DEFLATE_CONFIGS:
    if INPUT_DTYPE not in config.allowed_dtypes:
        print("Skipping config", config.name, "for unsupported dtype", INPUT_DTYPE)
        continue

    start_dt = datetime.now(timezone.utc)
    total_time_start = perf_counter()
    extra = BASE_EXTRA.copy()

    # Put in tmpfs to avoid disk I/O bottleneck
    # This is automatically deleted at end of scope/context manager
    with mem_temper.TemporaryDirectory() as temp_dir:
        temp_dir_path = Path(temp_dir)

        caller = lambda in_file, out_file: config.compressor[0](
            in_file, out_file, *config.compressor[1], **config.compressor[2]
        )
        # Run compression
        compress_start = perf_counter()
        if extra["n_jobs"] > 1:
            outputs = Parallel(n_jobs=extra["n_jobs"], prefer="processes")(
                delayed(caller)(input_file, temp_dir_path / (input_file.name))
                for input_file in INPUT_FILES
            )
        else:
            # Should maybe have option to disable where we don't invoke joblib overhead
            raise NotImplementedError("Parallelization is required for this test.")

        compress_time_seconds = perf_counter() - compress_start

        # Calculate sizes
        compressed_size = get_dir_size_bytes(temp_dir_path)
        compression_ratio = initial_size / compressed_size

    # Get pixel count for bitrate
    npixels = 0
    for input_file in INPUT_FILES:
        # Note that this does not actually read image data, so not super slow
        with TiffFile(input_file) as tif:
            for page in tif.pages:
                npixels += prod(page.shape)

    bitrate = 8 * initial_size / npixels
    bitrate_compressed = 8 * compressed_size / npixels

    space_saving = 1 - 1 / compression_ratio

    end_dt = datetime.now(timezone.utc)
    total_time_seconds = perf_counter() - total_time_start

    test = CompressionTest(
        name=config.get_name(),
        id=config.get_pathlike_str(),
        series=config.series,
        vary_var=config.vary_var,
        vary_value=config.vary_value,
        groupings=config.groupings,
        dataset_name=DATASET,
        dataset_path=DATASET_DIR,
        start_dt=start_dt,
        end_dt=end_dt,
        total_time_seconds=total_time_seconds,
        config=config,
        baseline_test=None,  # TODO: For now
        host=host,
        compress_time_seconds=compress_time_seconds,
        nbytes=initial_size,
        nbytes_stored=compressed_size,
        npixels=npixels,
        bitrate=bitrate,
        bitrate_compressed=bitrate_compressed,
        compression_ratio=compression_ratio,
        space_saving=space_saving,
        lossless=config.is_lossless,
        extra=extra,
    )

    tests.append(test)

    pprint(test.model_dump())

    output_file = TEST_OUTPUT_DIR / f"test_{test.name}.json"
    with open(output_file, "w") as f:
        f.write(test.model_dump_json(indent=4, fallback=repr))


{'baseline_test': None,
 'bitrate': 16.000252,
 'bitrate_compressed': 20.663912473282444,
 'compress_time_seconds': 20.683986539952457,
 'compression_ratio': 0.7743089320905536,
 'config': {'extra': {},
            'groupings': {'differencing': 'none',
                          'implementation': 'libtiff/tiffcp'},
            'is_lossless': True,
            'name': 'libtiff_lzw_pred{i}',
            'series': 'tiff-lzw',
            'vary_value': 1,
            'vary_var': 'predictor'},
 'dataset_name': 'Canislupus_CZM_712',
 'dataset_path': PosixPath('/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/datasets/Canislupus_CZM_712'),
 'end_dt': datetime.datetime(2026, 7, 29, 16, 15, 33, 422406, tzinfo=datetime.timezone.utc),
 'extra': {'n_jobs': 16},
 'groupings': {'differencing': 'none', 'implementation': 'libtiff/tiffcp'},
 'host': 'n-62-12-19',
 'id': '{"name":"libtiff_lzw_pred{i}","series":"tiff-lzw","vary_var":"predictor","vary_value":1,"groupings":{"implementati

In [ ]:
raise StopIteration()

StopIteration: 

In [ ]:
compressor_funcs = {
    "libtiff": libtiff.compress,
    "imagemagick": imagemagick,
}

N_JOBS = 8


initial_size = sum(f.stat().st_size for f in INPUT_FILES)
print(
    f"Initial size of all input files: {initial_size} bytes = {initial_size / (10**9):.2f} GB"
)

tests = []
i = 0
host = socket.gethostname()
for config in CONFIGS:
    for compressor, compressor_func in compressor_funcs.items():
        if (args := setup.get(compressor)) is None:
            print(f"Skipping {compressor} for {name} (not supported)")
            continue

    break

for name, setup in TESTS.items():
    i += 1
    if i == 1:
        continue
    for compressor, compressor_func in compressor_funcs.items():
        if (args := setup.get(compressor)) is None:
            print(f"Skipping {compressor} for {name} (not supported)")
            continue

        print(f"Running {compressor.upper()} for {name} with args {args}")

        total_start_time = perf_counter()
        with mem_temper.TemporaryDirectory() as temp_dir:
            output_dir = Path(temp_dir)

            # subprocess releases GIL so prefer threads for execution
            start_time = perf_counter()
            outputs = Parallel(n_jobs=N_JOBS, prefer="threads")(
                delayed(compressor_func)(
                    input_file, output_path=output_dir / (input_file.name), args=args
                )
                for input_file in INPUT_FILES
            )
            compress_time_seconds = perf_counter() - start_time

            nbytes_stored = get_dir_size_bytes(output_dir)
            print(outputs)

        extra = {"setup": "file-stack"}
        test = CompressionTest(
            name=name,
        )

        # key = name + "_" + compressor

        # output_path = OUTPUT_DIR / (key + ".tif")

        # with mem_temper.TemporaryDirectory() as temp_dir:
        #     temp_output_path = Path(temp_dir) / (key + ".tif")
        #     output_path = temp_output_path

        #     print(
        #         f"{compressor.upper()}: Compressing {input_file} with args {args} -> {name}"
        #     )

        #     try:
        #         _, result = compressor_func(
        #             input_file, output_path=output_path, args=args
        #         )

        #         compressed_size = output_path.stat().st_size
        #         test_result = TestResult(
        #             time=result.time,
        #             memory=result.memory,
        #             returncode=result.returncode,
        #             initial_size=initial_size,
        #             compressed_size=compressed_size,
        #             compression_ratio=initial_size / compressed_size,
        #         )

        #         test_results[key] = test_result

        #     except Exception as e:
        #         print(
        #             f"{compressor.upper()}: Error compressing {input_file} with args {args}: {e}"
        #         )

    break

Initial size of all input files: 12576198072 bytes = 12.58 GB
Running LIBTIFF for lzw_pred1 with args ['-c', 'lzw:1']
[(PosixPath('/dev/shm/tmpn4owdd8b/Canislupus_CZM_712_0809.tif'), ProcessResult(time=ProcessTiming(wall=0.16828187997452915, user=0.136606, system=0.010968), memory=ProcessMemoryUsage(max_rss=18948), returncode=0)), (PosixPath('/dev/shm/tmpn4owdd8b/Canislupus_CZM_712_0081.tif'), ProcessResult(time=ProcessTiming(wall=0.1672273820731789, user=0.13301, system=0.013000999999999999), memory=ProcessMemoryUsage(max_rss=18964), returncode=0)), (PosixPath('/dev/shm/tmpn4owdd8b/Canislupus_CZM_712_0650.tif'), ProcessResult(time=ProcessTiming(wall=0.1727831221651286, user=0.13328199999999998, system=0.016908), memory=ProcessMemoryUsage(max_rss=19004), returncode=0)), (PosixPath('/dev/shm/tmpn4owdd8b/Canislupus_CZM_712_0728.tif'), ProcessResult(time=ProcessTiming(wall=0.17885441216640174, user=0.128167, system=0.022674999999999997), memory=ProcessMemoryUsage(max_rss=18916), returncod

In [ ]:
@dataclass(frozen=True, slots=True)
class TestResult(ProcessResult):
    initial_size: int
    compressed_size: int
    compression_ratio: float

In [ ]:
# OUTPUT_DIR = input_file.parent / "compressed"
# OUTPUT_DIR.mkdir(exist_ok=True)
# test_results = {}
# compressor_funcs = {
#     "libtiff": libtiff.compress,
#     "imagemagick": imagemagick,
# }
# initial_size = input_file.stat().st_size

# for name, setup in TESTS.items():
#     for compressor in ["libtiff", "imagemagick"]:
#         if (args := setup.get(compressor)) is None:
#             print(f"Skipping {compressor} for {name} (not supported)")
#             continue

#         compressor_func = compressor_funcs[compressor]
#         key = name + "_" + compressor

#         output_path = OUTPUT_DIR / (key + ".tif")

#         with mem_temper.TemporaryDirectory() as temp_dir:
#             temp_output_path = Path(temp_dir) / (key + ".tif")
#             output_path = temp_output_path

#             print(
#                 f"{compressor.upper()}: Compressing {input_file} with args {args} -> {name}"
#             )

#             try:
#                 _, result = compressor_func(
#                     input_file, output_path=output_path, args=args
#                 )

#                 compressed_size = output_path.stat().st_size
#                 test_result = TestResult(
#                     time=result.time,
#                     memory=result.memory,
#                     returncode=result.returncode,
#                     initial_size=initial_size,
#                     compressed_size=compressed_size,
#                     compression_ratio=initial_size / compressed_size,
#                 )

#                 test_results[key] = test_result

#             except Exception as e:
#                 print(
#                     f"{compressor.upper()}: Error compressing {input_file} with args {args}: {e}"
#                 )